# 00b — Stitch images acquired with different beamstop positions

All images are aligned and intensity-normalized to one explicitly selected reference image per polarization. The fixed detector mask never moves. Each beamstop mask may be shifted independently. Their union is the `mask_pixel` used for registration, normalization, and stitching.

The reference image remains unchanged wherever it is valid. Other images only fill its masked region; multiple valid fill values are averaged.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

BASEFOLDER = Path.cwd().resolve()
sys.path.insert(0, str(BASEFOLDER / "library"))
from interactive import cimshow

from beamstop_stitching import shift_mask, stitch_images
from data_loading import Frame, SextantsNexusLoader, load_average
from image_preprocessing import fit_dark_frame, fit_horizontal_band, load_detector_masks

%matplotlib qt
print("Base folder:", BASEFOLDER)


## Select the input images

In [ ]:
# One position in these parallel lists defines one input to stitching.
# Every IMAGE_ID_GROUPS item may be one ID or a list of IDs to average first.
INPUT_KIND = "raw"  # "raw" or "preprocessed"
IMAGE_ID_GROUPS = [
    452,
    453,
    464,
    465,
    466,
    467,
    468,
    469,
    470,
    471,
]
POLARIZATIONS = ["+", "-", "+", "-", "+", "-", "+", "-", "+", "-"]

# Each raw input may use a different dark. A dark entry may also be a list.
# These entries are ignored when INPUT_KIND="preprocessed" because notebook 01a
# has already fitted and subtracted the dark from those files.
DARK_ID_GROUPS = [443, 443, 443, 443, 443, 443, 443, 443, 443, 443]

# The fixed detector mask is always mask_detector.png. Each raw input selects
# one mask_beamstop_<ID>.png, which can then be moved by the corresponding shift.
BEAMSTOP_MASK_IDS = [95, 95, 95, 95, 95, 95, 95, 95, 95, 95]
MASK_SHIFTS = [
    (0, 0), (0, 0),
    (52, -5), (52, -5),
    (-52, -5), (-52, -5),
    (0, -20), (0, -20),
    (-10, 5), (-10, 5),
]

RAW_FOLDER = Path("/nfs/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/")
PREPROCESSED_FOLDER = BASEFOLDER / "processed" / "cleaned_acquisitions"
MASK_FOLDER = BASEFOLDER / "processed" / "mask_pixels"
USER = "rb"

input_count = len(IMAGE_ID_GROUPS)
if not all(len(values) == input_count for values in (
    POLARIZATIONS, DARK_ID_GROUPS, BEAMSTOP_MASK_IDS, MASK_SHIFTS
)):
    raise ValueError("All parallel input lists must have the same length")
print("Fixed detector mask:", MASK_FOLDER / "mask_detector.png")
for index, values in enumerate(zip(
    IMAGE_ID_GROUPS, POLARIZATIONS, DARK_ID_GROUPS, BEAMSTOP_MASK_IDS, MASK_SHIFTS
)):
    image_ids, polarization, dark_ids, beamstop_id, shift = values
    print(f"input {index}: {polarization}, images={image_ids}, darks={dark_ids}, "
          f"mask_beamstop_{beamstop_id}.png, shift={shift}")


## Load and average every input group


In [ ]:
# Dark fitting always uses this corner and excludes the fixed detector mask.
DARK_FIT_ROWS = slice(0, 600)
DARK_FIT_COLUMNS = slice(0, 200)
DARK_FIT_PERCENTILE = 100
DARK_FIT_STRIDE = 1


def as_id_list(image_ids):
    if isinstance(image_ids, (list, tuple, np.ndarray)):
        return [int(image_id) for image_id in image_ids]
    return [int(image_ids)]


def average_preprocessed_files(image_ids):
    images = []
    detector_masks = []
    beamstop_masks = []
    energies = []
    sources = []
    for image_id in as_id_list(image_ids):
        input_file = PREPROCESSED_FOLDER / f"cleaned_ImId_{image_id:04d}_{USER}.npz"
        with np.load(input_file, allow_pickle=False) as saved:
            images.append(np.asarray(saved["image"], dtype=float))
            detector_masks.append(np.asarray(saved["mask_detector"], dtype=np.uint8))
            beamstop_masks.append(
                np.asarray(saved["mask_beamstop"], dtype=np.uint8)
            )
            energies.append(float(saved["energy_eV"]))
        sources.append(input_file)

    image_stack = np.stack(images)
    detector_mask = np.maximum.reduce(detector_masks).astype(np.uint8)
    beamstop_stack = np.stack(beamstop_masks).astype(np.uint8)
    pixel_mask_stack = np.clip(detector_mask[None, ...] + beamstop_stack, 0, 1)
    valid = (pixel_mask_stack == 0) & np.isfinite(image_stack)
    counts = valid.sum(axis=0)
    averaged = np.divide(
        np.where(valid, image_stack, 0).sum(axis=0),
        counts,
        out=np.zeros(image_stack.shape[1:], dtype=float),
        where=counts > 0,
    )
    beamstop_mask = (
        (counts == 0) & (detector_mask == 0)
    ).astype(np.uint8)
    pixel_mask = np.clip(detector_mask + beamstop_mask, 0, 1).astype(np.uint8)
    return averaged, detector_mask, beamstop_mask, pixel_mask, float(np.mean(energies)), sources


loader = SextantsNexusLoader(RAW_FOLDER)
frames = []
mask_detectors = []
unshifted_mask_beamstops = []
mask_beamstops = []
mask_pixels = []
group_names = []
group_id_lists = []
energies_eV = []
dark_fit_plots = []

for input_index, values in enumerate(zip(
    IMAGE_ID_GROUPS, DARK_ID_GROUPS, BEAMSTOP_MASK_IDS, MASK_SHIFTS
)):
    image_ids, dark_ids, beamstop_id, beamstop_shift = values
    image_id_list = as_id_list(image_ids)
    group_name = "+".join(str(image_id) for image_id in image_id_list)

    if INPUT_KIND == "raw":
        # load_average accepts one ID or a list and always returns a float average.
        loaded = load_average(loader, image_ids)
        raw_average = np.asarray(loaded.image, dtype=float)
        dark_average = np.asarray(load_average(loader, dark_ids).image, dtype=float)
        mask_detector, mask_beamstop, _ = load_detector_masks(
            MASK_FOLDER, beamstop_id, raw_average.shape
        )
        # Keep the PNG mask unchanged; apply the editable shift below.
        image, scale, offset, dark_values, fitted_pixels = fit_dark_frame(
            raw_average,
            dark_average,
            DARK_FIT_ROWS,
            DARK_FIT_COLUMNS,
            percentile=DARK_FIT_PERCENTILE,
            stride=DARK_FIT_STRIDE,
            mask=mask_detector,
        )
        image_values = raw_average[DARK_FIT_ROWS, DARK_FIT_COLUMNS][
            ::DARK_FIT_STRIDE, ::DARK_FIT_STRIDE
        ].ravel()
        dark_fit_plots.append((group_name, dark_values, image_values, fitted_pixels, scale, offset))
        exposure = float(loaded.exposure)
        energy_eV = float(loaded.metadata["energy_eV"])
        source_name = loaded.source
    elif INPUT_KIND == "preprocessed":
        image, mask_detector, mask_beamstop, mask_pixel, energy_eV, sources = (
            average_preprocessed_files(image_ids)
        )
        exposure = 1.0
        source_name = sources[0]
    else:
        raise ValueError('INPUT_KIND must be "raw" or "preprocessed"')

    mask_beamstop_unshifted = np.asarray(mask_beamstop, dtype=np.uint8)
    mask_beamstop = shift_mask(mask_beamstop_unshifted, beamstop_shift).astype(np.uint8)
    mask_pixel = np.clip(mask_detector + mask_beamstop, 0, 1).astype(np.uint8)

    frames.append(Frame(group_name, image, exposure, Path(source_name)))
    mask_detectors.append(np.asarray(mask_detector, dtype=np.uint8))
    unshifted_mask_beamstops.append(mask_beamstop_unshifted)
    mask_beamstops.append(mask_beamstop)
    mask_pixels.append(mask_pixel)
    group_names.append(group_name)
    group_id_lists.append(image_id_list)
    energies_eV.append(energy_eV)
    print(f"Loaded input {input_index}, IDs {image_id_list}: average shape {image.shape}")

energy_eV = float(np.mean(energies_eV))
print("Photon-energy range:", min(energies_eV), "to", max(energies_eV), "eV")


## Inspect fitted dark normalization and the separate masks


In [ ]:
if dark_fit_plots:
    fig, axes = plt.subplots(len(dark_fit_plots), 1, figsize=(7, 4 * len(dark_fit_plots)), squeeze=False)
    for axis, fit_data in zip(axes.flat, dark_fit_plots):
        group_name, dark_values, image_values, fitted_pixels, scale, offset = fit_data
        step = max(1, fitted_pixels.sum() // 5000)
        x = dark_values[fitted_pixels][::step]
        y = image_values[fitted_pixels][::step]
        axis.scatter(x, y, s=3, alpha=0.15, label="corner pixels")
        fit_x = np.linspace(x.min(), x.max(), 200)
        axis.plot(fit_x, scale * fit_x + offset, color="red", linewidth=2,
                  label=f"fit: {scale:.5g} x + {offset:.5g}")
        axis.set_title(f"input {group_name}: dark normalization")
        axis.set_xlabel("Dark intensity")
        axis.set_ylabel("Image intensity")
        axis.legend()
        axis.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

fig, axes = plt.subplots(len(IMAGE_ID_GROUPS), 3, figsize=(12, 3.5 * len(IMAGE_ID_GROUPS)), squeeze=False)
for row, group_name in enumerate(group_names):
    for axis, mask, title in zip(
        axes[row],
        (mask_detectors[row], mask_beamstops[row], mask_pixels[row]),
        ("fixed mask_detector", "shifted mask_beamstop", "combined mask_pixel"),
    ):
        axis.imshow(mask, cmap="gray", vmin=0, vmax=1)
        axis.set_title(f"input {group_name}: {title}")
        axis.set_axis_off()
plt.tight_layout()
plt.show()


## Adjust each beamstop-mask displacement

Select one input at a time. Move its red beamstop mask with the row and column
sliders until it covers the beamstop artifact in the image. The orange overlay
is the fixed detector mask. Slider changes are stored immediately in
`MASK_SHIFTS`, `mask_beamstops`, and `mask_pixels`, so later cells use exactly
what is displayed here.


In [ ]:
# The range and step are beside the widget that uses them.
MASK_SHIFT_RANGE = 100
MASK_SHIFT_STEP = 1
MASK_DISPLAY_PERCENTILES = (1, 99.9)


class MaskShiftWidget:
    """Inspect and move one beamstop mask at a time."""

    def __init__(self, images, detector_masks, beamstop_masks, initial_shifts, names):
        self.images = [np.asarray(image, dtype=float) for image in images]
        self.detector_masks = [np.asarray(mask, dtype=bool) for mask in detector_masks]
        self.beamstop_masks = [np.asarray(mask, dtype=bool) for mask in beamstop_masks]
        self.names = list(names)
        self.shifts = [tuple(map(float, shift)) for shift in initial_shifts]
        self._changing_input = False

        options = [(f"input {i}: IDs {name}", i) for i, name in enumerate(self.names)]
        self.input_widget = widgets.Dropdown(options=options, description="input")
        self.row_widget = widgets.FloatSlider(
            min=-MASK_SHIFT_RANGE, max=MASK_SHIFT_RANGE, step=MASK_SHIFT_STEP,
            description="row shift", continuous_update=False,
            layout=widgets.Layout(width="600px"),
        )
        self.column_widget = widgets.FloatSlider(
            min=-MASK_SHIFT_RANGE, max=MASK_SHIFT_RANGE, step=MASK_SHIFT_STEP,
            description="column shift", continuous_update=False,
            layout=widgets.Layout(width="600px"),
        )
        self.input_widget.observe(self._select_input, names="value")
        self.row_widget.observe(self._move_mask, names="value")
        self.column_widget.observe(self._move_mask, names="value")

        self.figure, self.axis = plt.subplots(figsize=(8, 7))
        self._select_input()
        display(widgets.VBox([self.input_widget, self.row_widget, self.column_widget]))
        plt.show()

    def _select_input(self, change=None):
        self._changing_input = True
        row_shift, column_shift = self.shifts[self.input_widget.value]
        self.row_widget.value = row_shift
        self.column_widget.value = column_shift
        self._changing_input = False
        self._draw()

    def _move_mask(self, change=None):
        if self._changing_input:
            return
        index = self.input_widget.value
        shift = (float(self.row_widget.value), float(self.column_widget.value))
        self.shifts[index] = shift
        mask_beamstops[index] = shift_mask(self.beamstop_masks[index], shift).astype(np.uint8)
        mask_pixels[index] = np.clip(
            self.detector_masks[index].astype(np.uint8) + mask_beamstops[index], 0, 1
        ).astype(np.uint8)
        self._draw()

    def _draw(self):
        index = self.input_widget.value
        image = self.images[index]
        finite = image[np.isfinite(image)]
        vmin, vmax = np.percentile(finite, MASK_DISPLAY_PERCENTILES)
        shifted = shift_mask(self.beamstop_masks[index], self.shifts[index])
        self.axis.clear()
        self.axis.imshow(image, cmap="viridis", vmin=vmin, vmax=vmax)
        self.axis.imshow(
            np.ma.masked_where(~self.detector_masks[index], self.detector_masks[index]),
            cmap="Oranges", vmin=0, vmax=1, alpha=0.35,
        )
        self.axis.imshow(
            np.ma.masked_where(~shifted, shifted), cmap="Reds", vmin=0, vmax=1, alpha=0.45
        )
        self.axis.set_title(
            f"input {index}, IDs {self.names[index]} | mask shift {self.shifts[index]}"
        )
        self.axis.set_axis_off()
        self.figure.canvas.draw_idle()


mask_shift_widget = MaskShiftWidget(
    [frame.image for frame in frames],
    mask_detectors,
    unshifted_mask_beamstops,
    MASK_SHIFTS,
    group_names,
)


In [ ]:
# Run this cell after the mask overlays look correct.
MASK_SHIFTS = list(mask_shift_widget.shifts)
print("Stored mask shifts:")
for input_index, shift in enumerate(MASK_SHIFTS):
    print(f"  input {input_index}, IDs {group_names[input_index]}: {shift}")


## Optionally correct the horizontal band, then stitch to each reference


In [ ]:
# The band algorithm is always available; disable it without deleting code.
CORRECT_HORIZONTAL_BAND = False
BAND_EDGE_COLUMNS = 20
BAND_SKIPPED_EDGE_COLUMNS = 0
BAND_POLYNOMIAL_ORDER = 2
BAND_CENTER = 1024
BAND_WIDTH = 80
BAND_EDGE = 11
SUBTRACT_POLYNOMIAL_BACKGROUND = False

if CORRECT_HORIZONTAL_BAND:
    corrected_frames = []
    for frame, mask_detector, mask_pixel in zip(frames, mask_detectors, mask_pixels):
        band_mask = mask_pixel.copy()
        band_mask[:20, :] = 1
        band_mask[-200:, :] = 1
        fit = fit_horizontal_band(
            frame.image,
            edge_columns=BAND_EDGE_COLUMNS,
            skipped_edge_columns=BAND_SKIPPED_EDGE_COLUMNS,
            polynomial_order=BAND_POLYNOMIAL_ORDER,
            band_center=BAND_CENTER,
            band_width=BAND_WIDTH,
            band_edge=BAND_EDGE,
            mask=band_mask,
        )
        corrected_image = frame.image - fit.band_image
        if SUBTRACT_POLYNOMIAL_BACKGROUND:
            corrected_image -= fit.polynomial_image
        corrected_frames.append(Frame(frame.image_id, corrected_image, frame.exposure, frame.source))

        fig, axis = plt.subplots(figsize=(9, 4))
        axis.plot(fit.rows, fit.measured_profile, label="measured")
        axis.plot(fit.rows, fit.fitted_profile, linewidth=2, label="complete fit")
        axis.plot(fit.rows, fit.polynomial_profile, "--", label=f"polynomial order {BAND_POLYNOMIAL_ORDER}")
        axis.fill_between(fit.rows, fit.polynomial_profile, fit.fitted_profile, alpha=0.25, label="band")
        axis.set_title(f"ID {frame.image_id}: horizontal-band fit")
        axis.legend()
        axis.grid(alpha=0.2)
        plt.tight_layout()
        plt.show()
    frames = corrected_frames



## Choose references and fitting parameters


In [ ]:
# Select the reference rows here, immediately before stitching.
PLUS_REFERENCE_INPUT = 0
MINUS_REFERENCE_INPUT = 1
for reference_input, polarization in (
    (PLUS_REFERENCE_INPUT, "+"), (MINUS_REFERENCE_INPUT, "-")
):
    if not 0 <= reference_input < len(IMAGE_ID_GROUPS):
        raise ValueError("Reference input index is outside IMAGE_ID_GROUPS")
    if POLARIZATIONS[reference_input] != polarization:
        raise ValueError(f"Reference input {reference_input} is not polarization {polarization}")

# Registration and intensity fitting use only this central mutually valid ROI.
FIND_IMAGE_SHIFTS = True
MAX_IMAGE_SHIFT = 10.0
REGISTRATION_UPSAMPLE = 10
FIT_ORDER = 1  # 1 = factor + offset; 2 or higher = polynomial mapping.
FIT_PERCENTILES = (2, 98)
DEFAULT_FIT_ROI = np.s_[800:-800, 800:-800]
FIT_ROIS = [DEFAULT_FIT_ROI for _ in frames]



## Inspect registration and intensity fits one input at a time

Choose an input and adjust the rectangular ROI. Press **Evaluate and store ROI**
to register that image to the first image of the same polarization and display
the aligned image, valid overlap, residual, and intensity scatter/fit. Both
combined `mask_pixel` arrays are always excluded from registration and fitting.
Each input keeps its own ROI in `FIT_ROIS`; the final stitching cell uses them.


In [ ]:
# Fit-widget display parameters live beside the widget.
FIT_WIDGET_PERCENTILES = (1, 99.9)
FIT_WIDGET_MAX_POINTS = 5000


class StitchFitWidget:
    """Evaluate and store one image-to-reference fit at a time."""

    def __init__(self):
        self.reference_for_input = {}
        for polarization, reference in (("+", PLUS_REFERENCE_INPUT), ("-", MINUS_REFERENCE_INPUT)):
            for index, value in enumerate(POLARIZATIONS):
                if value == polarization:
                    self.reference_for_input[index] = reference

        options = [
            (f"input {i}: {POLARIZATIONS[i]} IDs {group_names[i]}", i)
            for i in range(len(frames))
            if i != self.reference_for_input[i]
        ]
        self.input_widget = widgets.Dropdown(options=options, description="input")
        height, width = frames[0].image.shape
        default_rows = self._slice_bounds(DEFAULT_FIT_ROI[0], height)
        default_columns = self._slice_bounds(DEFAULT_FIT_ROI[1], width)
        self.rows_widget = widgets.IntRangeSlider(
            value=default_rows, min=0, max=height, step=1, description="rows",
            continuous_update=False, layout=widgets.Layout(width="700px"),
        )
        self.columns_widget = widgets.IntRangeSlider(
            value=default_columns, min=0, max=width, step=1, description="columns",
            continuous_update=False, layout=widgets.Layout(width="700px"),
        )
        self.evaluate_button = widgets.Button(
            description="Evaluate and store ROI", button_style="primary"
        )
        self.output = widgets.Output()
        self.input_widget.observe(self._load_stored_roi, names="value")
        self.evaluate_button.on_click(self.evaluate)
        display(widgets.VBox([
            self.input_widget, self.rows_widget, self.columns_widget,
            self.evaluate_button, self.output,
        ]))
        self._load_stored_roi()

    @staticmethod
    def _slice_bounds(value, length):
        start = 0 if value.start is None else value.start % length
        stop = length if value.stop is None else value.stop % length
        return (start, stop)

    def _load_stored_roi(self, change=None):
        index = self.input_widget.value
        height, width = frames[index].image.shape
        roi = FIT_ROIS[index]
        self.rows_widget.value = self._slice_bounds(roi[0], height)
        self.columns_widget.value = self._slice_bounds(roi[1], width)

    def evaluate(self, button=None):
        index = self.input_widget.value
        reference_index = self.reference_for_input[index]
        row_start, row_stop = self.rows_widget.value
        column_start, column_stop = self.columns_widget.value
        roi = np.s_[row_start:row_stop, column_start:column_stop]
        FIT_ROIS[index] = roi

        pair = stitch_images(
            [frames[reference_index], frames[index]],
            [mask_pixels[reference_index], mask_pixels[index]],
            register=FIND_IMAGE_SHIFTS,
            max_shift=MAX_IMAGE_SHIFT,
            upsample_factor=REGISTRATION_UPSAMPLE,
            fit_intensity=FIT_ORDER is not None,
            fit_degree=FIT_ORDER or 1,
            fit_percentiles=FIT_PERCENTILES,
            estimation_rois=[roi, roi],
            use_master_where_valid=False,
        )
        reference, item = pair.prepared_frames
        roi_mask = np.zeros(reference.image.shape, dtype=bool)
        roi_mask[roi] = True
        # The masks remain authoritative: ROI only further restricts valid overlap.
        fit_overlap = reference.valid & item.valid & roi_mask
        difference = np.where(fit_overlap, item.image - reference.image, np.nan)

        moving_values = item.fit_input[fit_overlap]
        reference_values = reference.image[fit_overlap]
        finite = np.isfinite(moving_values) & np.isfinite(reference_values)
        moving_values = moving_values[finite]
        reference_values = reference_values[finite]
        low, high = np.percentile(moving_values, FIT_PERCENTILES)
        selected = (moving_values >= low) & (moving_values <= high)
        step = max(1, selected.sum() // FIT_WIDGET_MAX_POINTS)

        with self.output:
            self.output.clear_output(wait=True)
            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            values = item.image[item.valid & np.isfinite(item.image)]
            vmin, vmax = np.percentile(values, FIT_WIDGET_PERCENTILES)
            residual_limit = np.nanpercentile(np.abs(difference), 99)
            axes[0].imshow(item.image, cmap="viridis", vmin=vmin, vmax=vmax)
            axes[0].set_title(f"aligned input | image shift {item.shift}")
            axes[1].imshow(fit_overlap, cmap="gray", vmin=0, vmax=1)
            axes[1].set_title(f"unmasked fit pixels in ROI: {fit_overlap.sum()}")
            axes[2].imshow(
                difference, cmap="coolwarm", vmin=-residual_limit, vmax=residual_limit
            )
            axes[2].set_title("residual in valid ROI")
            axes[3].scatter(
                moving_values[selected][::step], reference_values[selected][::step],
                s=3, alpha=0.2, label="unmasked ROI pixels",
            )
            fit_x = np.linspace(low, high, 300)
            axes[3].plot(
                fit_x, np.polyval(item.coefficients, fit_x), color="red", linewidth=2,
                label=f"order {len(item.coefficients) - 1}",
            )
            axes[3].set_xlabel("input intensity")
            axes[3].set_ylabel("reference intensity")
            axes[3].legend()
            for axis in axes[:3]:
                axis.set_axis_off()
            fig.suptitle(
                f"input {index} -> reference {reference_index} | coefficients {item.coefficients}"
            )
            plt.tight_layout()
            plt.show()
            print(f"Stored FIT_ROIS[{index}] = np.s_[{row_start}:{row_stop}, {column_start}:{column_stop}]")


fit_widget = StitchFitWidget()


## Run final stitching with the stored mask shifts and per-image fit ROIs


In [ ]:
stitch_results = []
stitched_image_ids = []
for polarization, reference_index in (
    ("+", PLUS_REFERENCE_INPUT),
    ("-", MINUS_REFERENCE_INPUT),
):
    selected = [index for index, value in enumerate(POLARIZATIONS) if value == polarization]
    if reference_index not in selected:
        raise ValueError(f"Reference input {reference_index} is not a {polarization} image")
    selected.remove(reference_index)
    selected.insert(0, reference_index)

    selected_frames = [frames[index] for index in selected]
    selected_masks = [mask_pixels[index] for index in selected]
    selected_fit_rois = [FIT_ROIS[index] for index in selected]
    result = stitch_images(
        selected_frames,
        selected_masks,
        register=FIND_IMAGE_SHIFTS,
        max_shift=MAX_IMAGE_SHIFT,
        upsample_factor=REGISTRATION_UPSAMPLE,
        fit_intensity=FIT_ORDER is not None,
        fit_degree=FIT_ORDER or 1,
        fit_percentiles=FIT_PERCENTILES,
        estimation_rois=selected_fit_rois,
        use_master_where_valid=True,
    )
    stitch_results.append(result)
    stitched_image_ids.append([group_id_lists[index] for index in selected])
    print(f"Polarization {polarization}; reference input {reference_index}, IDs {group_id_lists[reference_index]}")
    for item in result.prepared_frames:
        print(f"  ID {item.image_id}: shift={item.shift}, coefficients={item.coefficients}")


## Inspect alignment, normalization fits, and overlap


In [ ]:
DISPLAY_PERCENTILES = (1, 99.9)


def show_stored_fit(input_index):
    """Show one stored final fit; masks and that input's ROI remain excluded."""
    polarization = POLARIZATIONS[input_index]
    result = stitch_results[0 if polarization == "+" else 1]
    selected = [i for i, value in enumerate(POLARIZATIONS) if value == polarization]
    reference_index = PLUS_REFERENCE_INPUT if polarization == "+" else MINUS_REFERENCE_INPUT
    selected.remove(reference_index)
    selected.insert(0, reference_index)
    item = result.prepared_frames[selected.index(input_index)]
    reference = result.prepared_frames[0]
    roi_region = np.zeros(reference.image.shape, dtype=bool)
    roi_region[FIT_ROIS[input_index]] = True
    overlap = reference.valid & item.valid & roi_region
    difference = np.where(overlap, item.image - reference.image, np.nan)
    values = item.image[item.valid & np.isfinite(item.image)]
    vmin, vmax = np.percentile(values, DISPLAY_PERCENTILES)
    limit = np.nanpercentile(np.abs(difference), 99) if overlap.any() else 1

    moving_values = item.fit_input[overlap]
    reference_values = reference.image[overlap]
    finite = np.isfinite(moving_values) & np.isfinite(reference_values)
    moving_values, reference_values = moving_values[finite], reference_values[finite]
    low, high = np.percentile(moving_values, FIT_PERCENTILES)
    fitted = (moving_values >= low) & (moving_values <= high)
    step = max(1, fitted.sum() // 5000)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(item.image, cmap="viridis", vmin=vmin, vmax=vmax)
    axes[0].set_title(f"aligned input; shift {item.shift}")
    axes[1].imshow(overlap, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(f"unmasked overlap inside ROI: {overlap.sum()}")
    axes[2].imshow(difference, cmap="coolwarm", vmin=-limit, vmax=limit)
    axes[2].set_title("residual in fit pixels")
    axes[3].scatter(
        moving_values[fitted][::step], reference_values[fitted][::step],
        s=3, alpha=0.2, label="fit pixels",
    )
    fit_x = np.linspace(low, high, 300)
    axes[3].plot(fit_x, np.polyval(item.coefficients, fit_x), "r-", linewidth=2,
                 label=f"order {len(item.coefficients) - 1}")
    axes[3].set_xlabel("input intensity")
    axes[3].set_ylabel("reference intensity")
    axes[3].legend()
    for axis in axes[:3]:
        axis.set_axis_off()
    fig.suptitle(f"input {input_index}, IDs {group_names[input_index]} -> reference {reference_index}")
    plt.tight_layout()
    plt.show()


fit_result_widget = widgets.Dropdown(
    options=[(f"input {i}: IDs {name}", i) for i, name in enumerate(group_names)],
    description="inspect",
)
widgets.interact(show_stored_fit, input_index=fit_result_widget)


## Inspect the final combined images


In [ ]:
for polarization, image_ids, result in zip(("+", "-"), stitched_image_ids, stitch_results):
    image_values = result.image[result.missing_mask == 0]
    vmin, vmax = np.percentile(image_values, (1, 99.9))
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    shown = axes[0].imshow(result.image, cmap="viridis", vmin=vmin, vmax=vmax)
    axes[0].set_title(f"{polarization}: final stitched image")
    fig.colorbar(shown, ax=axes[0], label="Normalized intensity")
    axes[1].imshow(result.source_count, cmap="viridis")
    axes[1].set_title("number of contributing images")
    axes[2].imshow(result.missing_mask, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("final mask_pixel")
    for axis in axes:
        axis.set_axis_off()
    fig.suptitle(f"Reference IDs {image_ids[0]}; input groups {image_ids}")
    plt.tight_layout()
    plt.show()


## Simple names for interactive inspection

The names below are intentionally stable. Use them directly in later cells; no knowledge of the stitching loops is required.


In [ ]:
pos_reference = np.asarray(frames[PLUS_REFERENCE_INPUT].image, dtype=float)
neg_reference = np.asarray(frames[MINUS_REFERENCE_INPUT].image, dtype=float)
mask_pos_reference = np.asarray(mask_pixels[PLUS_REFERENCE_INPUT], dtype=np.uint8)
mask_neg_reference = np.asarray(mask_pixels[MINUS_REFERENCE_INPUT], dtype=np.uint8)

pos = np.asarray(stitch_results[0].image, dtype=float)
neg = np.asarray(stitch_results[1].image, dtype=float)
mask_pos = np.asarray(stitch_results[0].missing_mask, dtype=np.uint8)
mask_neg = np.asarray(stitch_results[1].missing_mask, dtype=np.uint8)


def show_image(image, title="image", percentiles=(1, 99.9)):
    """Display any 2-D NumPy image with a useful linear intensity range."""
    image = np.asarray(image, dtype=float)
    finite = image[np.isfinite(image)]
    vmin, vmax = np.percentile(finite, percentiles)
    fig, axis = plt.subplots(figsize=(6, 5))
    shown = axis.imshow(image, cmap="viridis", vmin=vmin, vmax=vmax)
    axis.set_title(title)
    axis.set_axis_off()
    fig.colorbar(shown, ax=axis, label="Intensity")
    plt.tight_layout()
    plt.show()
    return fig, axis


def show_input(input_number):
    """Display one averaged, dark-corrected stitching input and its mask."""
    show_image(frames[input_number].image, f"input {input_number}: IDs {group_id_lists[input_number]}")


print("Easy NumPy names: pos_reference, neg_reference, pos, neg")
print("Easy mask names: mask_pos_reference, mask_neg_reference, mask_pos, mask_neg")
print("Examples: show_input(2) or show_image(pos, 'stitched positive')")


## Save files for 01_FTH.ipynb

In [ ]:
OUTPUT_FOLDER = BASEFOLDER / "processed" / "stitched"
OUTPUT_NAME = "moved_beamstop"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

def roi_as_bounds(roi, shape):
    """Store a NumPy ROI as explicit (row_start, row_stop, column_start, column_stop)."""
    row_start, row_stop, _ = roi[0].indices(shape[0])
    column_start, column_stop, _ = roi[1].indices(shape[1])
    return row_start, row_stop, column_start, column_stop


output_files = []
for polarization, image_ids, result in zip(("+", "-"), stitched_image_ids, stitch_results):
    name = "plus" if polarization == "+" else "minus"
    output_file = OUTPUT_FOLDER / f"stitched_{OUTPUT_NAME}_{name}_{USER}.npz"
    fit_coefficients = np.asarray([item.coefficients for item in result.prepared_frames], dtype=float)
    # Keep both kinds of displacement explicit: the user moves beamstop masks,
    # while registration moves complete images and their masks to the reference.
    reference_input = PLUS_REFERENCE_INPUT if polarization == "+" else MINUS_REFERENCE_INPUT
    selected_inputs = [i for i, value in enumerate(POLARIZATIONS) if value == polarization]
    selected_inputs.remove(reference_input)
    selected_inputs.insert(0, reference_input)
    mask_shifts = np.asarray([MASK_SHIFTS[i] for i in selected_inputs], dtype=float)
    image_shifts = np.asarray([item.shift for item in result.prepared_frames], dtype=float)
    fit_roi_bounds = np.asarray(
        [roi_as_bounds(FIT_ROIS[i], result.image.shape) for i in selected_inputs],
        dtype=int,
    )
    factors = fit_coefficients[:, 0] if FIT_ORDER == 1 else np.full(len(image_ids), np.nan)
    offsets = fit_coefficients[:, -1]

    # Detector defects are fixed. The remaining missing region is the final beamstop mask.
    mask_detector = np.asarray(mask_detectors[0], dtype=np.uint8)
    mask_beamstop = (
        (result.missing_mask > 0) & (mask_detector == 0)
    ).astype(np.uint8)
    mask_pixel = np.clip(mask_detector + mask_beamstop, 0, 1).astype(np.uint8)

    reference_ids = image_ids[0]
    reference_exposure = frames[reference_input].exposure
    np.savez_compressed(
        output_file,
        image=result.image,
        mask_detector=mask_detector,
        mask_beamstop=mask_beamstop,
        mask_pixel=mask_pixel,
        source_count=result.source_count,
        reference_exposure=float(reference_exposure),
        reference_image_id=int(reference_ids[0]),
        reference_image_ids=np.asarray(reference_ids, dtype=int),
        ordered_input_groups=np.asarray(["+".join(map(str, ids)) for ids in image_ids]),
        polarization=np.asarray(polarization),
        mask_shifts=mask_shifts,
        image_shifts=image_shifts,
        fit_roi_bounds=fit_roi_bounds,
        fit_order=-1 if FIT_ORDER is None else FIT_ORDER,
        fit_coefficients=fit_coefficients,
        factors=factors,
        offsets=offsets,
        energy_eV=energy_eV,
    )
    output_files.append(output_file)
    print(f"Saved {polarization}: {output_file}")

print("Use these as PLUS_STITCHED_FILE and MINUS_STITCHED_FILE in 01_FTH:")
for polarization, output_file in zip(("+", "-"), output_files):
    print(f"  {polarization}: {output_file.relative_to(BASEFOLDER)}")


In [ ]:
print("image ID groups:", IMAGE_ID_GROUPS)
print("dark ID groups:", DARK_ID_GROUPS if INPUT_KIND == "raw" else "already subtracted")
print("plus reference input:", PLUS_REFERENCE_INPUT, group_id_lists[PLUS_REFERENCE_INPUT])
print("minus reference input:", MINUS_REFERENCE_INPUT, group_id_lists[MINUS_REFERENCE_INPUT])
print("stored mask shifts:", MASK_SHIFTS)
print("stored fit ROI bounds:")
for input_index, roi in enumerate(FIT_ROIS):
    print(f"  input {input_index}:", roi_as_bounds(roi, frames[input_index].image.shape))
